In [ ]:
# --- Deep learning framework ---
import timm
import torch
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from torchvision import transforms
from torch.utils.data import DataLoader

# --- Data processing ---
import os
import numpy as np
import pandas as pd
from pathlib import Path

# --- Image processing ---
from PIL import ImageFile, Image
from skimage import io, img_as_float32

# --- Spatial transcriptomics ---
import scanpy as sc
import anndata as ad

## 1. Load UNI2 (ViT-Giant) Model

In [ ]:
# UNI2 model config — ViT-Giant with register tokens
# embed_dim=1536, depth=24, num_heads=24, dynamic_img_size=True
timm_kwargs = {
    'model_name': 'vit_giant_patch14_224',
    'img_size': 224,
    'patch_size': 14,
    'depth': 24,
    'num_heads': 24,
    'init_values': 1e-5,
    'embed_dim': 1536,
    'mlp_ratio': 2.66667 * 2,
    'num_classes': 0,                        # No classification head, output raw features
    'no_embed_class': True,
    'mlp_layer': timm.layers.SwiGLUPacked,
    'act_layer': torch.nn.SiLU,
    'reg_tokens': 8,
    'dynamic_img_size': True
}

model = timm.create_model(pretrained=False, **timm_kwargs)

In [ ]:
# Set device
device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Load local pretrained weights
model.load_state_dict(
    torch.load("./UNI2.bin", map_location="cpu"),
    strict=True
)

In [ ]:
# Image transform pipeline — resize to 224x224, normalize with ImageNet stats
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
])

model.to(device).eval()

## 2. Batch Patch Processing Function

Extracts square patches centered at each spot from the full-slide image, applies the transform pipeline, and returns GPU-batched tensors.

In [ ]:
def process_image_batch(centers, im, r, batch_size=32):
    """
    Batch-process image patches centered at given coordinates.

    Args:
        centers: list of (x, y) center coordinates
        im: np.ndarray, full-slide H&E image
        r: int, half-size of the square patch in pixels
        batch_size: int, patches per GPU batch

    Returns:
        all_patches: list of torch.Tensor batches
    """
    n_patches = len(centers)
    all_patches = []

    for start_idx in range(0, n_patches, batch_size):
        end_idx = min(start_idx + batch_size, n_patches)
        batch_patches = []

        for i in range(start_idx, end_idx):
            center = centers[i]
            x, y = center
            patch = im[(x - r):(x + r), (y - r):(y + r), :]
            patch = np.transpose(patch, (1, 0, 2))
            patch = Image.fromarray(patch[:, :, [0, 1, 2]])
            patch_tensor = transform(patch)
            batch_patches.append(patch_tensor)

        batch_tensor = torch.stack(batch_patches).to(device)
        all_patches.append(batch_tensor)

    return all_patches

---
## 3. Leave-One-Out Validation (her2st)

Commented template — adjust paths and uncomment to run.

In [ ]:
# --- Configuration ---
Image.MAX_IMAGE_PIXELS = 100000000
exp_dir = r'./her2st/data/ST-cnts'
img_dir = r'./her2st/data/ST-imgs'
pos_dir = r'./her2st/data/ST-spotfiles'
feaure_dir = r'./her2st/data/ST-UNI2-features'
r = 56  # half-size → 112x112 patches

names = [file for file in os.listdir(exp_dir) if file.endswith('.tsv')]
names.sort()
names = [i[:2] for i in names]

def get_img(img_dir, name):
    pre = img_dir + '/' + name[0] + '/' + name
    fig_name = [file for file in os.listdir(pre) if file.endswith('.jpg')][0]
    path = pre + '/' + fig_name
    im = Image.open(path)
    return im

def get_meta(exp_dir, pos_dir, name):
    exp_path = exp_dir + '/' + name + '.tsv'
    adata = sc.read(exp_path, delimiter='\t')
    pos_path = pos_dir + '/' + name + '_selection.tsv'
    df = pd.read_csv(pos_path, sep='\t')
    x = np.around(df['x'].values).astype(int)
    y = np.around(df['y'].values).astype(int)
    id = [str(x[i]) + 'x' + str(y[i]) for i in range(len(x))]
    df.index = id
    adata.obs = adata.obs.merge(df, how='left', left_index=True, right_index=True)
    return adata

img_dict = {i: np.array(get_img(img_dir, i)) for i in names}
exp_dict = {i: get_meta(exp_dir, pos_dir, i) for i in names}

for name in names:
    im = img_dict[name]
    im = np.transpose(im, (1, 0, 2))
    centers = (np.floor(exp_dict[name].obs[['pixel_x', 'pixel_y']]).values).astype(int)
    patch_batches = process_image_batch(centers, im, r, batch_size=32)
    all_features = []
    with torch.no_grad():
        for batch in patch_batches:
            features = model(batch)          # [batch, 1536]
            all_features.append(features.cpu())
    patches_features = torch.cat(all_features, dim=0)
    print(f"Processed {name}: {patches_features.shape}")
    torch.save(patches_features, f'{feaure_dir}/{name}.pt')